In [ ]:
#@title Installing Java version 17 due to Neo4j requirements
!apt-get install openjdk-17-jre-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java
!java -version

openjdk version "17.0.12" 2024-07-16
OpenJDK Runtime Environment (build 17.0.12+7-Ubuntu-1ubuntu222.04)
OpenJDK 64-Bit Server VM (build 17.0.12+7-Ubuntu-1ubuntu222.04, mixed mode, sharing)


In [ ]:
#@title Installing Neo4J server on Google colab
!wget -q https://neo4j.com/artifact.php?name=neo4j-community-5.21.0-unix.tar.gz -O neo4j.tar.gz
# decompress and rename
!tar -xf neo4j.tar.gz  # or --strip-components=1
!mv neo4j-community-5.21.0 nj
# download apoc plugins and enable apoc command
!wget -q https://github.com/neo4j/apoc/releases/download/5.21.0/apoc-5.21.0-core.jar -O nj/plugins/apoc-5.21.0-core.jar
!echo "dbms.security.procedures.unrestricted=apoc.*" >> nj/conf/neo4j.conf
!echo "dbms.security.procedures.allowlist=apoc.*" >> nj/conf/neo4j.conf
# disable password, and start server
!sed -i '/#dbms.security.auth_enabled/s/^#//g' nj/conf/neo4j.conf
# start Neo4j server
!nj/bin/neo4j start

Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:1301). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
#@title Install Mongo, Neo4j and Azure OpenAI python engines
%%capture
!pip install pymongo==4.6.1 -q
!pip install py2neo==2021.2.4 -q
!pip install openai==1.40.6 -q
!pip install llama-index==0.10.65 -q
!pip install llama-index-llms-azure-openai==0.1.10 -q
!pip install llama-index-embeddings-azure-openai==0.1.11 -q
!pip install llama-index-vector-stores-neo4jvector==0.1.6 -q
!pip install llama-index-graph-stores-neo4j==0.2.14 -q
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from google.colab import userdata
from llama_index.core import Settings

llm = AzureOpenAI(
    engine="gpt-4o",
    deployment_name=userdata.get("AZURE_MODEL_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
    temperature=0.0001,
)

embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small",
    deployment_name=userdata.get("AZURE_EMBEDDING_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
)
llm.model = userdata.get("AZURE_MODEL_NAME")
embed_model.model_name = userdata.get("AZURE_EMBEDDING_NAME")
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 1024 # 1024 256 512 1536

In [ ]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir="/content/", required_exts=[".pdf"])
documents = reader.load_data(num_workers=-1)

In [ ]:
for doc in documents:
    print(doc.metadata['file_name'])

Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23-TESLA.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf
Q23ALPHABET.pdf


In [ ]:
# Enable async python function in the Jupyer notebook
import nest_asyncio
nest_asyncio.apply()

In [ ]:
#@title Create a pointer to Neo4j Property Graph Index
from llama_index.graph_stores.neo4j import Neo4jPGStore

username=""
password=""
url = "bolt://localhost:7687"
# url="bolt://34.239.240.152:7687"

# Neo4j Community version only support the default database
database="neo4j"

graph_store = Neo4jPGStore(
    username=username,
    password=password,
    url=url,
    # database=database
)

In [ ]:
# CLEAR DATABASE
from py2neo import Graph

graph = Graph(url, auth=(username, password))

query = """
MATCH (m)-[t]->(r)
DETACH DELETE m,t,r
"""
graph.run(query).to_table()
query = """
MATCH (m)
DETACH DELETE m
"""
graph.run(query).to_table()

In [ ]:
# @title Comprehensive Entities and Relations Specifications for Financial Analysis
from typing import Literal, Dict, List

# Define entities
entities = Literal[
    "COMPANY",         # e.g., Tesla
    "FIGURE",          # e.g., Specific financial figures within a table
    "QUARTER",         # e.g., Q1 2023, Q2 2023
    "SECTION",         # e.g., Summary, Financial Highlights
    "INDUSTRY",        # e.g., Automotive, Energy
    "REGION",          # e.g., North America, Europe
]

# Define relations
relations = Literal[
    "BELONGS_TO",       # Relation between FIGURE and TABLE
    "CONTAINS",         # Relation between TABLE and COLUMN_HEADER/FIGURE
    "MENTIONS",         # Relation between SECTION and FIGURE/ENTITY
    "IS_MEASURED_IN",   # Relation between FIGURE and UNIT/CURRENCY
    "REPORTED_ON",      # Relation between COMPANY and DATE
    "LISTS",            # Relation between COMPANY and STOCK_SYMBOL
    "OPERATES_IN",      # Relation between COMPANY and REGION/INDUSTRY
    "FOOTNOTE_OF",      # Relation between FOOTNOTE and TABLE/FIGURE
    "HAS_COLUMN",       # Relation between TABLE and COLUMN_HEADER
    "HAS_QUARTER",      # Relation between COMPANY and QUARTER
    "COVERS",           # Relation between QUARTER and TABLE
]

# Define validation schema for entities and their possible relations
validation_schema =  {
    "COMPANY": [
        "REPORTED_ON",
        "LISTS",
        "OPERATES_IN",
        "HAS_QUARTER",
    ],
    "FIGURE": [
        "BELONGS_TO",
        "IS_MEASURED_IN",
    ],
    "QUARTER": [
        "COVERS",
    ],
    "SECTION": [
        "MENTIONS",
    ],
    "INDUSTRY": [
        "OPERATES_IN",
    ],
    "REGION": [
        "OPERATES_IN",
    ],
}

In [ ]:
#@title Create the Entities(nodes), Relations(edges) and their Chunks and Embedding
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
from llama_index.core import PropertyGraphIndex

extract_prompt = """Give the following text, extract the knowledge graph according to the provided schema.
Try to limit to the output {max_triplets_per_chunk} extracted paths.
 Do not use abbrevations or include version name/number, just use full names according to the context.
 Only exist two companies in the text: ALPHABET and TESLA
 All QUARTER entities names must be this format 'Qnumber YYYY' such as 'Q4 2024'
 Always try to avoid generating duplicates entities names
-------
{text}
-------"""

# llama_index extractor
kg_extractor = SchemaLLMPathExtractor(
    llm=llm,
    extract_prompt=extract_prompt,
    possible_entities=entities,
    possible_relations=relations,
    kg_validation_schema=validation_schema,
    strict=True, # # if false, allows values outside of spec
    num_workers=4,
    max_triplets_per_chunk=10, # number of triplets extracted per chunk: 3, 5 10, 15
)

# Use the LLM and embed models to create the graph
index = PropertyGraphIndex.from_documents(
    documents,
    kg_extractors=[kg_extractor],
    llm=llm,
    embed_model=embed_model,
    property_graph_store=graph_store,
    show_progress=True,
)

Parsing nodes:   0%|          | 0/40 [00:00<?, ?it/s]


Extracting paths from text with schema: 100%|██████████| 42/42 [00:21<00:00,  1.91it/s]

Generating embeddings: 100%|██████████| 5/5 [00:00<00:00,  6.06it/s]

Generating embeddings: 100%|██████████| 18/18 [00:00<00:00, 26.22it/s]


In [ ]:
#@title Reconnect to Neo4j to check if the database was generated
from py2neo import Graph
url = "bolt://localhost:7687"
# url="bolt://34.239.240.152:7687"/
graph = Graph(url, auth=(username, password))

In [ ]:
#@title Retrieve all node labels
node_labels_query = """
CALL db.labels()
YIELD label
RETURN label
"""

node_labels = graph.run(node_labels_query).data()

def get_node_attributes(label):
    query = f"""
    MATCH (n:{label})
    RETURN keys(n) AS attributes
    LIMIT 1
    """
    result = graph.run(query).data()
    if result:
        return result[0]['attributes']
    else:
        return []

node_attributes = {}
for node in node_labels:
    attributes = get_node_attributes(node['label'])
    node_attributes[node['label']] = attributes

print("Node with their Attributes:")
for label, attributes in node_attributes.items():
    print(f"Label: {label} has Attributes: {attributes}")

Node with their Attributes:
Label: __Node__ has Attributes: ['page_label', '_node_type', 'embedding', 'file_size', '_node_content', 'file_type', 'doc_id', 'document_id', 'last_modified_date', 'creation_date', 'ref_doc_id', 'file_path', 'text', 'file_name', 'id']
Label: __Entity__ has Attributes: ['triplet_source_id', 'embedding', 'name', 'file_size', 'file_type', 'page_label', 'last_modified_date', 'creation_date', 'file_path', 'id', 'file_name']
Label: Chunk has Attributes: ['page_label', '_node_type', 'embedding', 'file_size', '_node_content', 'file_type', 'doc_id', 'document_id', 'last_modified_date', 'creation_date', 'ref_doc_id', 'file_path', 'text', 'file_name', 'id']
Label: QUARTER has Attributes: ['triplet_source_id', 'page_label', 'embedding', 'name', 'creation_date', 'file_type', 'file_name', 'file_size', 'file_path', 'id', 'last_modified_date']
Label: SECTION has Attributes: ['triplet_source_id', 'page_label', 'embedding', 'name', 'creation_date', 'file_type', 'file_name', '

In [ ]:
graph.run("""MATCH (n) RETURN DISTINCT n.name""").to_table()

n.name
null
TESLA
Q1 2023
Q1 2023 Update
Q4 2022
Q1 2022
$23.3B
$22.4B
"421,371 Model 3/Y production"
"3,889 Storage deployed"


In [ ]:
#@title Retrieve all relationship types
relationship_types_query = """
CALL db.relationshipTypes()
YIELD relationshipType
RETURN relationshipType
"""

relationship_types = graph.run(relationship_types_query).data()


def get_relationship_attributes(relationship_type):
    query = f"""
    MATCH ()-[r:{relationship_type}]->()
    RETURN keys(r) AS attributes
    LIMIT 1
    """
    result = graph.run(query).data()
    if result:
        return result[0]['attributes']
    else:
        return []

relationship_attributes = {}
for record in relationship_types:
    relationship_type = record['relationshipType']
    attributes = get_relationship_attributes(relationship_type)
    relationship_attributes[relationship_type] = attributes


print("Relationship Types with their Attributes:")
for relationship_type, attributes in relationship_attributes.items():
    print(f"Relationship Type: {relationship_type}, and Attributes: {attributes}")

Relationship Types with their Attributes:
Relationship Type: MENTIONS, and Attributes: []
Relationship Type: REPORTED_ON, and Attributes: ['triplet_source_id', 'file_type', 'page_label', 'last_modified_date', 'creation_date', 'file_name', 'file_size', 'file_path']
Relationship Type: HAS_QUARTER, and Attributes: ['triplet_source_id', 'file_type', 'page_label', 'last_modified_date', 'creation_date', 'file_name', 'file_size', 'file_path']
Relationship Type: IS_MEASURED_IN, and Attributes: ['triplet_source_id', 'file_type', 'page_label', 'last_modified_date', 'creation_date', 'file_name', 'file_size', 'file_path']
Relationship Type: OPERATES_IN, and Attributes: ['triplet_source_id', 'file_type', 'page_label', 'last_modified_date', 'creation_date', 'file_name', 'file_size', 'file_path']


In [ ]:
print(graph.run("""MATCH (n)-[r]->()
WITH labels(n) as NodeLabels, collect(DISTINCT type(r)) as Relationships
WHERE NOT NodeLabels[-1] IN ['Chunk', 'Entity', 'Node']
RETURN DISTINCT NodeLabels[-1] AS Node, Relationships
""").to_table())

 Node    | Relationships                                                   
---------|-----------------------------------------------------------------
 COMPANY | ['REPORTED_ON', 'HAS_QUARTER', 'IS_MEASURED_IN', 'OPERATES_IN'] 
 SECTION | ['MENTIONS']                                                    
 FIGURE  | ['IS_MEASURED_IN']                                              
 QUARTER | ['REPORTED_ON', 'IS_MEASURED_IN']                               



In [ ]:
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core import PropertyGraphIndex

graph_store = Neo4jPropertyGraphStore(
    username=username,
    password=password,
    url=url
)
index = PropertyGraphIndex.from_existing(
    property_graph_store=graph_store,
    llm=llm,
    embed_model=embed_model,
)

In [ ]:
# !rm -rf /content/storage
# index.storage_context.persist(persist_dir="/content/storage")

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

query = "How do Tesla's vehicle production and pricing strategies compare with Alphabet's service and product offerings in terms of market positioning and competitive advantage?"
# query = "List the finantial measures of Tesla and Alphabet and their sources"
retriever = index.as_retriever(
    include_text=False,
    similarity_top_k=3,
)
nodes = retriever.retrieve(query)
display(nodes)
print("\n\n")
query_engine = index.as_query_engine(
    include_text=False,
    similarity_top_k=3,
)
response = query_engine.query(query)
display(response)
print("\n\n")

print(response.response)

[NodeWithScore(node=TextNode(id_='8cc1837e-09c9-44c3-87e3-6fa006c4449f', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='a7ddf084-ec58-48a1-8cb3-b044b3d01128', node_type=None, metadata={}, hash=None)}, text="TESLA ({'creation_date': '2024-09-03', 'last_modified_date': '2024-09-03', 'file_size': 5935644, 'file_path': '/content/Q23-TESLA.pdf', 'name': 'TESLA', 'file_name': 'Q23-TESLA.pdf', 'page_label': '29', 'triplet_source_id': 'a7ddf084-ec58-48a1-8cb3-b044b3d01128', 'file_type': 'application/pdf'}) -> IS_MEASURED_IN ({'creation_date': '2024-09-03', 'last_modified_date': '2024-09-03', 'file_size': 5935644, 'file_path': '/content/Q23-TESLA.pdf', 'file_name': 'Q23-TESLA.pdf', 'page_label': '5', 'triplet_source_id': '2d50e27e-b96f-43fe-9475-5553722d8886', 'file_type': 'application/pdf'}) -> $22.4B ({'creation_date': '2024-09-03', 'last_modified_date': '2024-09-03', 'file_si

Response(response="Tesla's vehicle production and pricing strategies focus on innovation, high performance, and premium pricing, which position the company as a leader in the electric vehicle market. This approach emphasizes advanced technology, sustainability, and a strong brand image, providing a competitive advantage through differentiation and perceived value.\n\nIn contrast, Alphabet's service and product offerings, such as its search engine, advertising platforms, and various tech services, are designed to capture a broad user base through accessibility, scalability, and integration into daily life. Alphabet leverages its vast data resources and technological infrastructure to maintain a competitive edge, focusing on user experience, targeted advertising, and continuous innovation.\n\nBoth companies utilize their unique strengths to secure market positioning: Tesla through its cutting-edge automotive technology and premium market segment, and Alphabet through its expansive digita




Tesla's vehicle production and pricing strategies focus on innovation, high performance, and premium pricing, which position the company as a leader in the electric vehicle market. This approach emphasizes advanced technology, sustainability, and a strong brand image, providing a competitive advantage through differentiation and perceived value.

In contrast, Alphabet's service and product offerings, such as its search engine, advertising platforms, and various tech services, are designed to capture a broad user base through accessibility, scalability, and integration into daily life. Alphabet leverages its vast data resources and technological infrastructure to maintain a competitive edge, focusing on user experience, targeted advertising, and continuous innovation.

Both companies utilize their unique strengths to secure market positioning: Tesla through its cutting-edge automotive technology and premium market segment, and Alphabet through its expansive digital ecosystem and data

In [ ]:
from llama_index.core.indices.property_graph import (
    PGRetriever,
    VectorContextRetriever,
    LLMSynonymRetriever,
)

sub_retrievers = [
    VectorContextRetriever(index.property_graph_store, similarity_top_k=3),
    LLMSynonymRetriever(index.property_graph_store, ...),
]

retriever = PGRetriever(sub_retrievers=sub_retrievers)
display(retriever.retrieve(query))
print("\n\n")


query_engine = index.as_query_engine(sub_retrievers=[retriever])
response = query_engine.query(query)
print("\n\n")
display(response.response)

[NodeWithScore(node=TextNode(id_='a7ddf084-ec58-48a1-8cb3-b044b3d01128', embedding=None, metadata={'page_label': '29', 'file_name': 'Q23-TESLA.pdf', 'file_path': '/content/Q23-TESLA.pdf', 'file_type': 'application/pdf', 'file_size': 5935644, 'creation_date': '2024-09-03', 'last_modified_date': '2024-09-03'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='78bdf1db-47ab-431f-848f-0526597197f7', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '29', 'file_name': 'Q23-TESLA.pdf', 'file_path': '/content/Q23-TESLA.pdf', 'file_type': 'application/pdf', 'file_size': 5935644, 'creation_date': '2024-09-03', 'last_modified_date': '2024-09-03'}, hash='2f709d17022f874f8896c1ef290e9fbf10564ef2940eb5349d6e1f

"The provided information does not include specific details about Tesla's vehicle production and pricing strategies or Alphabet's service and product offerings. Therefore, a comparison in terms of market positioning and competitive advantage cannot be made based on the given context."

In [ ]:
from llama_index.core.indices.property_graph import LLMSynonymRetriever

prompt = (
    "Given some initial query, generate synonyms or related keywords up to {max_keywords} in total, "
    "considering possible cases of capitalization, pluralization, common expressions, etc.\n"
    "Provide all synonyms/keywords separated by '^' symbols: 'keyword1^keyword2^...'\n"
    "Note, result should be in one-line, separated by '^' symbols."
    "----\n"
    "QUERY: {query_str}\n"
    "----\n"
    "KEYWORDS: "
)


def parse_fn(output: str) -> list[str]:
    matches = output.strip().split("^")

    # capitalize to normalize with ingestion
    return [x.strip().capitalize() for x in matches if x.strip()]


synonym_retriever = LLMSynonymRetriever(
    index.property_graph_store,
    llm=llm,
    # include source chunk text with retrieved paths
    include_text=False,
    synonym_prompt=prompt,
    output_parsing_fn=parse_fn,
    max_keywords=10,
    # the depth of relations to follow after node retrieval
    path_depth=1,
)

retriever = index.as_retriever(sub_retrievers=[synonym_retriever])
display(retriever.retrieve(query))
print("\n\n")


query_engine = index.as_query_engine(sub_retrievers=[synonym_retriever])
response = query_engine.query(query)
print("\n\n")
display(response.response)

[]

'Empty Response'

In [ ]:
from llama_index.core.indices.property_graph import VectorContextRetriever

vector_retriever = VectorContextRetriever(
    index.property_graph_store,
    # only needed when the graph store doesn't support vector queries
    # vector_store=index.vector_store,
    embed_model=embed_model,
    # include source chunk text with retrieved paths
    include_text=False,
    # the number of nodes to fetch
    similarity_top_k=3,
    # the depth of relations to follow after node retrieval
    path_depth=1,
    # # can provide any other kwargs for the VectorStoreQuery class
    # ...,
)

retriever = index.as_retriever(sub_retrievers=[vector_retriever])
display(retriever.retrieve(query))
print("\n\n")


query_engine = index.as_query_engine(sub_retrievers=[vector_retriever])
response = query_engine.query(query)
display(response.response)

[NodeWithScore(node=TextNode(id_='44f20b65-e616-4f12-a78b-ad7c58bf4259', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='a7ddf084-ec58-48a1-8cb3-b044b3d01128', node_type=None, metadata={}, hash=None)}, text="TESLA ({'creation_date': '2024-09-03', 'last_modified_date': '2024-09-03', 'file_size': 5935644, 'file_path': '/content/Q23-TESLA.pdf', 'name': 'TESLA', 'file_name': 'Q23-TESLA.pdf', 'page_label': '29', 'triplet_source_id': 'a7ddf084-ec58-48a1-8cb3-b044b3d01128', 'file_type': 'application/pdf'}) -> IS_MEASURED_IN ({'creation_date': '2024-09-03', 'last_modified_date': '2024-09-03', 'file_size': 5935644, 'file_path': '/content/Q23-TESLA.pdf', 'file_name': 'Q23-TESLA.pdf', 'page_label': '5', 'triplet_source_id': '2d50e27e-b96f-43fe-9475-5553722d8886', 'file_type': 'application/pdf'}) -> $22.4B ({'creation_date': '2024-09-03', 'last_modified_date': '2024-09-03', 'file_si

"Tesla's vehicle production and pricing strategies focus on innovation, high performance, and premium pricing, which position the company as a leader in the electric vehicle market. This approach emphasizes advanced technology, sustainability, and brand prestige, giving Tesla a competitive advantage in attracting environmentally conscious and tech-savvy consumers.\n\nIn contrast, Alphabet's service and product offerings, such as its search engine, advertising services, and various technology products, are designed to be widely accessible and integrated into everyday life. Alphabet leverages its vast data resources and technological expertise to provide highly personalized and efficient services, which enhances user experience and loyalty.\n\nBoth companies leverage their unique strengths to maintain competitive advantages: Tesla through its cutting-edge automotive technology and brand appeal, and Alphabet through its extensive data capabilities and ubiquitous service offerings."

In [ ]:
index.property_graph_store.get_schema_str()

'Node properties:\nChunk {page_label: STRING, _node_type: STRING, embedding: LIST, file_size: INTEGER, _node_content: STRING, file_type: STRING, doc_id: STRING, document_id: STRING, last_modified_date: STRING, creation_date: STRING, ref_doc_id: STRING, file_path: STRING, text: STRING, file_name: STRING, id: STRING}\nQUARTER {triplet_source_id: STRING, page_label: STRING, embedding: LIST, name: STRING, creation_date: STRING, file_type: STRING, file_name: STRING, file_size: INTEGER, file_path: STRING, id: STRING, last_modified_date: STRING}\nSECTION {triplet_source_id: STRING, page_label: STRING, embedding: LIST, name: STRING, creation_date: STRING, file_type: STRING, file_name: STRING, file_size: INTEGER, file_path: STRING, id: STRING, last_modified_date: STRING}\nCOMPANY {triplet_source_id: STRING, embedding: LIST, name: STRING, file_size: INTEGER, file_type: STRING, page_label: STRING, last_modified_date: STRING, creation_date: STRING, file_path: STRING, id: STRING, file_name: STRING}

In [ ]:
from llama_index.core.indices.property_graph import TextToCypherRetriever

DEFAULT_RESPONSE_TEMPLATE = (
    "Generated Cypher query(be concise must not include triple backticks):\n{query}\n\n" "Cypher Response:\n{response}"
)
DEFAULT_ALLOWED_FIELDS = ["text", "label", "type"]

DEFAULT_TEXT_TO_CYPHER_TEMPLATE = (
    index.property_graph_store.text_to_cypher_template,
)


cypher_retriever = TextToCypherRetriever(
    index.property_graph_store,
    # customize the LLM, defaults to Settings.llm
    llm=llm,
    # customize the text-to-cypher template.
    # Requires `schema` and `question` template args
    # text_to_cypher_template=DEFAULT_TEXT_TO_CYPHER_TEMPLATE,
    # customize how the cypher result is inserted into
    # a text node. Requires `query` and `response` template args
    response_template=DEFAULT_RESPONSE_TEMPLATE,
    # an optional callable that can clean/verify generated cypher
    cypher_validator=None,
    # allowed fields in the resulting
    allowed_output_field=DEFAULT_ALLOWED_FIELDS,
)

# cypher_retriever.retrieve(str_or_query_bundle={"query_str":query})
retriever = index.as_retriever(sub_retrievers=[cypher_retriever])
display(retriever.retrieve(query))
print("\n\n")


query_engine = index.as_query_engine(sub_retrievers=[ cypher_retriever ])
response = query_engine.query(query)
display(response.response)


[NodeWithScore(node=TextNode(id_='7602f7b9-58a2-4ad2-a499-75252e37f627', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text="Generated Cypher query(be concise must not include triple backticks):\nMATCH (t:COMPANY {name: 'TESLA'})-[:REPORTED_ON]->(tq:QUARTER)-[:IS_MEASURED_IN]->(tf:FIGURE),\n      (a:COMPANY {name: 'ALPHABET'})-[:REPORTED_ON]->(aq:QUARTER)-[:IS_MEASURED_IN]->(af:FIGURE)\nRETURN t.name AS Tesla, tq.name AS TeslaQuarter, tf.name AS TeslaFigure, \n       a.name AS Alphabet, aq.name AS AlphabetQuarter, af.name AS AlphabetFigure\n\nCypher Response:\n[{'Tesla': 'TESLA', 'TeslaQuarter': 'Q1 2023', 'TeslaFigure': 'Net income $ 15,051', 'Alphabet': 'ALPHABET', 'AlphabetQuarter': 'Q1 2023', 'AlphabetFigure': 'Revenues $ 69,787'}, {'Tesla': 'TESLA', 'TeslaQuarter': 'Q1 2023', 'TeslaFigure': 'Net income $ 15,051', 'Alphabet': 'ALPHABET', 'AlphabetQuarter': 'Q1 2022', 'AlphabetFigure': '$68,011'}, {'Tesla': 'TESLA', 'T

"Tesla's vehicle production and pricing strategies focus on high-performance electric vehicles with a premium pricing model, aiming to position itself as a leader in the electric vehicle market. This strategy leverages advanced technology, innovation, and brand prestige to create a competitive advantage.\n\nOn the other hand, Alphabet's service and product offerings, such as its search engine, advertising services, and cloud computing, are designed to cater to a broad audience with a focus on accessibility, scalability, and integration across various platforms. Alphabet's competitive advantage lies in its vast data resources, technological infrastructure, and ability to innovate in digital services.\n\nIn summary, Tesla's market positioning is centered around premium electric vehicles and technological innovation in the automotive industry, while Alphabet focuses on providing comprehensive digital services and products that leverage its technological prowess and extensive data capabili

In [ ]:
# NOTE: current v1 is needed
from pydantic.v1 import BaseModel, Field
from llama_index.core.indices.property_graph import CypherTemplateRetriever

# write a query with template params
cypher_query = """
MATCH (c:Chunk)-[:MENTIONS]->(o)
WHERE o.name IN $names
RETURN c.text, o.name, o.label;
"""


# create a pydantic class to represent the params for our query
# the class fields are directly used as params for running the cypher query
class TemplateParams(BaseModel):
    """Template params for a cypher query."""

    names: list[str] = Field(
        description="A list of entity names or keywords to use for lookup in a knowledge graph."
    )


template_retriever = CypherTemplateRetriever(
    index.property_graph_store, TemplateParams, cypher_query
)

query_engine = index.as_query_engine(sub_retrievers=[ template_retriever ])
response = query_engine.query(query)
display(response.response)


"There is no specific information provided to compare Tesla's vehicle production and pricing strategies with Alphabet's service and product offerings in terms of market positioning and competitive advantage."

In [ ]:
from typing import Tuple
prompt = (
    "Some text is provided below. Given the text, extract up to "
    "{max_paths_per_chunk} "
    "knowledge triples in the form of `subject,predicate,object` on each line. Avoid stopwords.\n"
)


def parse_fn(response_str: str) -> List[Tuple[str, str, str]]:
    lines = response_str.split("\n")
    triples = [line.split(",") for line in lines]
    return triples


kg_extractor = SimpleLLMPathExtractor(
    llm=llm,
    extract_prompt=prompt,
    parse_fn=parse_fn,
)

In [ ]:
#@title Create Custom LLamaIndex Retriver Engine to make prompt queries
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core import PropertyGraphIndex

graph_store = Neo4jPropertyGraphStore(
    username="",
    password="",
    url="bolt://localhost:7687",
)
index = PropertyGraphIndex.from_existing(
    property_graph_store=graph_store,
    llm=llm,
    embed_model=embed_model,
)
custom_sub_retriever = CompaniesQ1Retriever(
    index.property_graph_store,
    include_text=True,
    vector_store=index.vector_store,
    embed_model=embed_model
)

query_engine = RetrieverQueryEngine.from_args(
#    index.as_retriever(sub_retrievers=[custom_sub_retriever]), llm=llm
#    index.as_retriever(sub_retrievers=[custom_sub_retriever],similarity_top_k=25), llm=llm
    index.as_retriever(similarity_top_k=3)
)

In [ ]:
# less than 1 seconds
# This query creates a vector index for nodes labeled as `__Entity__` if it does not already exist.
# The index is created on the `embedding` property of these nodes.
# The index configuration specifies:
# - `vector.dimensions`: The dimension of the vectors, set to 1536.
# - `vector.similarity_function`: The function used to compare vectors, set to 'cosine'.

graph_store.structured_query("""
CREATE VECTOR INDEX entity IF NOT EXISTS
FOR (m:`__Entity__`)
ON m.embedding
OPTIONS {indexConfig: {
 `vector.dimensions`: 1536,
 `vector.similarity_function`: 'cosine'
}}
""")

[]

In [ ]:
# # Define the similarity threshold for vector comparison
# similarity_threshold = 0.9

# # Define the word edit distance threshold for string comparison
# word_edit_distance = 5

# # Execute a structured query on the graph store
# # This query performs the following steps:
# # 1. Matches all nodes labeled as __Entity__.
# # 2. Uses the vector index 'entity' to find the 10 nearest neighbors based on the 'embedding' property.
# # 3. Filters the nodes based on a similarity score above the threshold, name similarity, or edit distance.
# # 4. Ensures the nodes have the same labels.
# # 5. Orders the filtered nodes by their name and collects them.
# # 6. Collects the names of the nodes and performs additional filtering to union intersecting results.
# # 7. Applies extra filtering to remove duplicate sets.
# # 8. Returns the final combined results.

# data = graph_store.structured_query("""
# MATCH (e:__Entity__)
# WHERE NOT 'QUARTER' IN labels(e) AND NOT 'DATE' IN labels(e) AND NOT 'FIGURE' IN labels(e)
# CALL {
#   WITH e
#   CALL db.index.vector.queryNodes('entity', 10, e.embedding)
#   YIELD node, score
#   WITH node, score
#   WHERE score > toFLoat($cutoff)
#       AND (toLower(node.name) CONTAINS toLower(e.name) OR toLower(e.name) CONTAINS toLower(node.name)
#            OR apoc.text.distance(toLower(node.name), toLower(e.name)) < $distance)
#       AND labels(e) = labels(node)
#       AND NOT 'QUARTER' IN labels(node) AND NOT 'DATE' IN labels(node) AND NOT 'FIGURE' IN labels(node)
#   WITH node, score
#   ORDER BY node.name
#   RETURN collect(node) AS nodes
# }
# WITH distinct nodes
# WHERE size(nodes) > 1
# WITH collect([n in nodes | n.name]) AS results
# UNWIND range(0, size(results)-1, 1) as index
# WITH results, index, results[index] as result
# WITH apoc.coll.sort(reduce(acc = result, index2 IN range(0, size(results)-1, 1) |
#         CASE WHEN index <> index2 AND
#             size(apoc.coll.intersection(acc, results[index2])) > 0
#             THEN apoc.coll.union(acc, results[index2])
#             ELSE acc
#         END
# )) as combinedResult
# WITH distinct(combinedResult) as combinedResult
# // extra filtering
# WITH collect(combinedResult) as allCombinedResults
# UNWIND range(0, size(allCombinedResults)-1, 1) as combinedResultIndex
# WITH allCombinedResults[combinedResultIndex] as combinedResult, combinedResultIndex, allCombinedResults
# WHERE NOT any(x IN range(0,size(allCombinedResults)-1,1)
#     WHERE x <> combinedResultIndex
#     AND apoc.coll.containsAll(allCombinedResults[x], combinedResult)
# )
# RETURN combinedResult
# """, param_map={'cutoff': similarity_threshold, 'distance': word_edit_distance})

# # Print the results of the query
# for row in data:
#     print(row)

{'combinedResult': ['q1 2022', 'q1 2023', 'q2 2022']}
{'combinedResult': ['15', '17', '2023', 'april 19 2023', 'january 31 2023', 'quarter 3 2023']}
{'combinedResult': ['total revenue', 'total revenues']}
{'combinedResult': ['cars', 'days', 'gwh', 'mw']}
{'combinedResult': ['billion dollars', 'dollar']}
{'combinedResult': ['depreciation amortization and impairment', 'depreciation amortization impairment']}
{'combinedResult': ['alphabet', 'alphabet inc']}
{'combinedResult': ['april 20, 2023', 'april 25 2023']}
{'combinedResult': ['march 31 2022', 'march 31 2023']}
{'combinedResult': ['109120', '69300']}
{'combinedResult': ['constant currency revenues', 'non-gaap constant currency revenues']}
{'combinedResult': ['quarter ended march 31 2021', 'quarter ended march 31 2022']}


In [ ]:
# relationship_types_query = """
# MATCH (c)
# WHERE c.name = '109120'
# RETURN c.name, labels(c)
# """
# graph.run(relationship_types_query)

c.name,labels(c)
109120,"['__Node__', '__Entity__', 'FIGURE']"


In [ ]:
# # Execute a structured query on the graph store
# # This query performs the following steps:
# # 1. Matches all nodes labeled as __Entity__.
# # 2. Uses the vector index 'entity' to find the 10 nearest neighbors based on the 'embedding' property.
# # 3. Filters the nodes based on a similarity score above the threshold, name similarity, or edit distance.
# # 4. Ensures the nodes have the same labels.
# # 5. Orders the filtered nodes by their name and collects them.
# # 6. Collects the names of the nodes and performs additional filtering to union intersecting results.
# # 7. Applies extra filtering to remove duplicate sets.
# # 8. For each combined result, unwinds the names, matches the corresponding __Entity__ nodes, and orders them by the size of their name (preferring longer names).
# # 9. Merges the nodes using apoc.refactor.mergeNodes and discards conflicting properties.
# # 10. Returns the count of merged nodes.

# graph_store.structured_query("""
# MATCH (e:__Entity__)
# WHERE NOT 'QUARTER' IN labels(e) AND NOT 'DATE' IN labels(e) AND NOT 'FIGURE' IN labels(e)
# CALL {
#   WITH e
#   CALL db.index.vector.queryNodes('entity', 10, e.embedding)
#   YIELD node, score
#   WITH node, score
#   WHERE score > toFLoat($cutoff)
#       AND (toLower(node.name) CONTAINS toLower(e.name) OR toLower(e.name) CONTAINS toLower(node.name)
#            OR apoc.text.distance(toLower(node.name), toLower(e.name)) < $distance)
#       AND labels(e) = labels(node)
#       AND NOT 'QUARTER' IN labels(node) AND NOT 'DATE' IN labels(node)
#   WITH node, score
#   ORDER BY node.name
#   RETURN collect(node) AS nodes
# }
# WITH distinct nodes
# WHERE size(nodes) > 1
# WITH collect([n in nodes | n.name]) AS results
# UNWIND range(0, size(results)-1, 1) as index
# WITH results, index, results[index] as result
# WITH apoc.coll.sort(reduce(acc = result, index2 IN range(0, size(results)-1, 1) |
#         CASE WHEN index <> index2 AND
#             size(apoc.coll.intersection(acc, results[index2])) > 0
#             THEN apoc.coll.union(acc, results[index2])
#             ELSE acc
#         END
# )) as combinedResult
# WITH distinct(combinedResult) as combinedResult
# // extra filtering
# WITH collect(combinedResult) as allCombinedResults
# UNWIND range(0, size(allCombinedResults)-1, 1) as combinedResultIndex
# WITH allCombinedResults[combinedResultIndex] as combinedResult, combinedResultIndex, allCombinedResults
# WHERE NOT any(x IN range(0,size(allCombinedResults)-1,1)
#     WHERE x <> combinedResultIndex
#     AND apoc.coll.containsAll(allCombinedResults[x], combinedResult)
# )
# CALL {
#   WITH combinedResult
#   UNWIND combinedResult AS name
#   MATCH (e:__Entity__ {name:name})
#   WHERE NOT 'QUARTER' IN labels(e) AND NOT 'DATE' IN labels(e) AND NOT 'FIGURE' IN labels(e)
#   WITH e
#   ORDER BY size(e.name) DESC // prefer longer names to remain after merging
#   RETURN collect(e) AS nodes
# }
# CALL apoc.refactor.mergeNodes(nodes, {properties: {
#     `.*`: 'discard'
# }})
# YIELD node
# RETURN count(*)
# """, param_map={'cutoff': similarity_threshold, 'distance': word_edit_distance})

[{'count(*)': 10}]

In [ ]:
from IPython.display import Markdown, display
r = graph.run("""MATCH (n)
WHERE NOT n:Chunk
RETURN labels(n)[-1] as classes,n.name as names
""").to_data_frame()

entity_classes = list(set(r['classes'].values.tolist()))
entity_names = r['names'].values.tolist()
Markdown(r.to_markdown())

|     | classes       | names                                                    |
|----:|:--------------|:---------------------------------------------------------|
|   0 | QUARTER       | q1 2023                                                  |
|   1 | DATE          | 2023                                                     |
|   2 | SECTION       | update                                                   |
|   3 | COMPANY       | tesla                                                    |
|   4 | QUARTER       | q1                                                       |
|   5 | METRIC        | operating margin                                         |
|   6 | COLUMN_HEADER | net income                                               |
|   7 | QUARTER       | first quarter                                            |
|   8 | METRIC        | total automotive revenues                                |
|   9 | CURRENCY      | millions                                                 |
|  10 | DATE          | q1 2022                                                  |
|  11 | METRIC        | adjusted ebitda                                          |
|  12 | CURRENCY      | usd                                                      |
|  13 | METRIC        | net income gaap                                          |
|  14 | METRIC        | free cash flow                                           |
|  15 | METRIC        | total revenue                                            |
|  16 | COLUMN_HEADER | operating income                                         |
|  17 | METRIC        | cash equivalents                                         |
|  18 | DATE          | q2 2022                                                  |
|  19 | METRIC        | physical footprint                                       |
|  20 | UNIT          | locations globally                                       |
|  21 | METRIC        | cumulative miles driven                                  |
|  22 | METRIC        | fsd beta miles                                           |
|  23 | METRIC        | energy storage deployments                               |
|  24 | UNIT          | gwh                                                      |
|  25 | METRIC        | solar deployments                                        |
|  26 | UNIT          | mw                                                       |
|  27 | METRIC        | services and other revenue                               |
|  28 | METRIC        | volume                                                   |
|  29 | UNIT          | cars                                                     |
|  30 | SECTION       | operating system                                         |
|  31 | METRIC        | low sga costs                                            |
|  32 | METRIC        | low lead time                                            |
|  33 | SECTION       | tooling for cybertruck production line                   |
|  34 | DATE          | 15                                                       |
|  35 | METRIC        | vehicle deliveries                                       |
|  36 | UNIT          | millions of units                                        |
|  37 | METRIC        | operating cash flow                                      |
|  38 | UNIT          | billion dollars                                          |
|  39 | CURRENCY      | dollars                                                  |
|  40 | INDUSTRY      | auto industry                                            |
|  41 | SECTION       | financial statements                                     |
|  42 | DATE          | quarter 3 2023                                           |
|  43 | DATE          | q3 2022                                                  |
|  44 | METRIC        | net income per share                                     |
|  45 | CURRENCY      | dollar                                                   |
|  46 | DATE          | august 2022                                              |
|  47 | TABLE         | total assets                                             |
|  48 | METRIC        | goodwill and intangible assets                           |
|  49 | TABLE         | total liabilities                                        |
|  50 | METRIC        | deferred revenue                                         |
|  51 | TABLE         | total stockholders equity                                |
|  52 | METRIC        | noncontrolling interests in subsidiaries                 |
|  53 | FIGURE        | 30632                                                    |
|  54 | FIGURE        | 34085                                                    |
|  55 | TABLE         | total liabilities and equity                             |
|  56 | FIGURE        | 66038                                                    |
|  57 | METRIC        | net cash used in investing activities                    |
|  58 | TABLE         | reconciliation of gaap to non gaap financial information |
|  59 | COLUMN_HEADER | shares used in eps calculation diluted                   |
|  60 | COLUMN_HEADER | net income attributable to common stockholders           |
|  61 | COLUMN_HEADER | interest expense                                         |
|  62 | UNIT          | millions of usd                                          |
|  63 | METRIC        | depreciation amortization impairment                     |
|  64 | UNIT          | millions usd                                             |
|  65 | METRIC        | stock based compensation expense                         |
|  66 | METRIC        | capital expenditures                                     |
|  67 | METRIC        | provision for income taxes                               |
|  68 | DATE          | april 19 2023                                            |
|  69 | METRIC        | days payable outstanding                                 |
|  70 | UNIT          | days                                                     |
|  71 | METRIC        | days of supply                                           |
|  72 | SECTION       | forward looking statements                               |
|  73 | DATE          | january 31 2023                                          |
|  74 | COMPANY       | alphabet                                                 |
|  75 | DATE          | april 25 2023                                            |
|  76 | DATE          | april 20, 2023                                           |
|  77 | SECTION       | blog post                                                |
|  78 | TABLE         | consolidated financial results                           |
|  79 | COLUMN_HEADER | revenues                                                 |
|  80 | METRIC        | google cloud                                             |
|  81 | UNIT          | number of employees                                      |
|  82 | COMPANY       | deepmind                                                 |
|  83 | DATE          | first quarter 2023                                       |
|  84 | METRIC        | employee severance charges                               |
|  85 | CURRENCY      | usd 2.0 billion                                          |
|  86 | DATE          | january 2023                                             |
|  87 | METRIC        | useful lives assessment                                  |
|  88 | METRIC        | stock repurchases                                        |
|  89 | DATE          | december 31 2022                                         |
|  90 | SECTION       | annual report                                            |
|  91 | DATE          | march 31 2023                                            |
|  92 | SECTION       | quarterly report                                         |
|  93 | SECTION       | press release                                            |
|  94 | TABLE         | total current liabilities                                |
|  95 | FIGURE        | 69300                                                    |
|  96 | FIGURE        | 109120                                                   |
|  97 | FIGURE        | 256144                                                   |
|  98 | COMPANY       | alphabet inc                                             |
|  99 | DATE          | march 31 2022                                            |
| 100 | TABLE         | segment results                                          |
| 101 | COMPANY       | google services                                          |
| 102 | COMPANY       | other bets                                               |
| 103 | TABLE         | other income expense                                     |
| 104 | COLUMN_HEADER | interest income                                          |
| 105 | COLUMN_HEADER | foreign currency exchange gain loss                      |
| 106 | METRIC        | non-gaap constant currency revenues                      |
| 107 | METRIC        | constant currency revenues                               |
| 108 | CURRENCY      | foreign exchange rate                                    |
| 109 | REGION        | united states                                            |
| 110 | SECTION       | total revenues prior year comparative periods            |
| 111 | QUARTER       | quarter ended march 31 2021                              |
| 112 | QUARTER       | quarter ended march 31 2022                              |
| 113 | METRIC        | revenues excluding hedging effect                        |
| 114 | CURRENCY      | $ 55,423                                                 |

In [ ]:
#@title Install libs to display Neo4j Graph
!pip install yfiles_jupyter_graphs_for_neo4j -q
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
#@title Load Visualization Component
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget
from neo4j import GraphDatabase
driver = GraphDatabase.driver(uri = url,)
g = Neo4jGraphWidget(driver)

In [ ]:
#@title Generate a backup of neo4j database
!nj/bin/neo4j stop
!nj/bin/neo4j-admin database dump neo4j --to-path=/content/

Stopping Neo4j............ stopped.
2024-08-22 16:00:49.119+0000 INFO  [o.n.c.d.DumpCommand] Starting dump of database 'neo4j'
Done: 44 files, 260.1MiB processed.
2024-08-22 16:00:51.147+0000 INFO  [o.n.c.d.DumpCommand] Dump completed successfully


In [ ]:
# Check the /content/

In [ ]:
!mkdir /content/neo4j_companies_2023_q1/
!mv /content/neo4j.dump /content/neo4j_companies_2023_q1/neo4j.dump

In [ ]:
!nj/bin/neo4j-admin database load neo4j --from-path=/content/neo4j_companies_2023_q1/ --overwrite-destination=true

Done: 44 files, 260.1MiB processed.


In [ ]:
!nj/bin/neo4j start

Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:23890). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
graph = Graph(url, auth=(username, password))

In [ ]:
# @title Prompt attributes for user's queries
from pydantic import BaseModel
from typing import Optional, List

# Must use names for Neo4J identification
class Entities(BaseModel):
    """List of named entities in the text such as names of FINANCIAL_STATEMENTS, PRODUCTS, ASSETS, LOSSES, PROFITS, SERVICES, EXPENSES, COMPANIES, CEOS"""
    names: Optional[List[str]]

# Prompt to extract entities from the existing graph
prompt_template_entities = """
Check carefully the following performance:
Extract all named entities such as names of financial staments, products, assets, profits, expenses, companies or ceos
from the following text:
{text}
"""

In [ ]:
print(prompt_template_entities)


Check carefully the following performance:
Extract all named entities such as names of financial staments, products, assets, profits, expenses, companies or ceos
from the following text:
{text}



In [ ]:
#@title Create Custom LLamaIndex Retriver Class
from typing import Any, Optional

from llama_index.core.embeddings import BaseEmbedding
from llama_index.core.retrievers import CustomPGRetriever, VectorContextRetriever
from llama_index.core.vector_stores.types import VectorStore
from llama_index.program.openai import OpenAIPydanticProgram


class CompaniesQ1Retriever(CustomPGRetriever):
    """Custom retriever with cohere reranking."""

    def init(
        self,
        embed_model: Optional[BaseEmbedding] = None,
        vector_store: Optional[VectorStore] = None,
        similarity_top_k: int = 20,
        path_depth: int = 2,
        include_text: bool = True,
        **kwargs: Any,
    ) -> None:
        """Uses any kwargs passed in from class constructor."""
        self.entity_extraction = OpenAIPydanticProgram.from_defaults(
            output_cls=Entities, prompt_template_str=prompt_template_entities
        )
        self.vector_retriever = VectorContextRetriever(
            self.graph_store,
            include_text=self.include_text,
            embed_model=embed_model,
            similarity_top_k=similarity_top_k,
            path_depth=path_depth,
        )

    def custom_retrieve(self, query_str: str) -> str:
        """Define custom retriever with reranking.

        Could return `str`, `TextNode`, `NodeWithScore`, or a list of those.
        """
        entities = self.entity_extraction(text=query_str).names
        result_nodes = []
        if entities:
            print(f"Detected entities: {entities}")
            for entity in entities:
                result_nodes.extend(self.vector_retriever.retrieve(entity))
        else:
            result_nodes.extend(self.vector_retriever.retrieve(query_str))
        final_text = "\n\n".join(
            [n.get_content(metadata_mode="llm") for n in result_nodes]
        )
        return final_text

In [ ]:
#@title Create Custom LLamaIndex Retriver Engine to make prompt queries
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core import PropertyGraphIndex

graph_store = Neo4jPropertyGraphStore(
    username="",
    password="",
    url="bolt://localhost:7687",
)
index = PropertyGraphIndex.from_existing(
    property_graph_store=graph_store,
    llm=llm,
    embed_model=embed_model,
)
custom_sub_retriever = CompaniesQ1Retriever(
    index.property_graph_store,
    include_text=True,
    vector_store=index.vector_store,
    embed_model=embed_model
)

query_engine = RetrieverQueryEngine.from_args(
#    index.as_retriever(sub_retrievers=[custom_sub_retriever]), llm=llm
#    index.as_retriever(sub_retrievers=[custom_sub_retriever],similarity_top_k=25), llm=llm
    index.as_retriever(similarity_top_k=3)
)

In [ ]:
response = query_engine.query("What was the percentage change in adjusted EBITDA margin from Q2-2022 to Q3-2022 of Tesla?")
print(str(response))

The adjusted EBITDA margin for Tesla decreased from 22.2% in Q2-2022 to 18.3% in Q3-2022. This represents a percentage change of -3.9%.


In [ ]:
# Can add system prompting to guide the model to call functions and perform in specific ways
system_prompt = f"""
Assistant is a helpful assistant that helps users get answers to questions about a Neo4jGraph.
Assistant has access to several tools and sometimes you may need to call multiple tools in sequence to get answers for your users.

"""
next_messages = [
    {
        "role": "system",
        "content": system_prompt,
    }]

In [ ]:
from openai import AzureOpenAI
from google.colab import userdata
import json
from pprint import pprint

client = AzureOpenAI(
    api_key=userdata.get('AZURE_API_KEY'),
    api_version=userdata.get('AZURE_API_VERSION'),
    azure_endpoint=userdata.get('AZURE_BASE_URL')
)

import concurrent.futures

BLUE,GREEN,MANGETA,RESET = '\033[94m', '\033[92m', '\033[95m', '\033[0m'
def dict_to_expression_string(dictionary):
    return ' '.join(f'{BLUE}{key}{RESET}={GREEN}{value}{RESET}' for key, value in dictionary.items())

def run_multiturn_conversation(messages, tools, available_functions):
    response = client.chat.completions.create(
        messages=messages,
        tools=tools,
        tool_choice="auto",
        model=userdata.get('AZURE_MODEL_NAME'),
        temperature=1e-4,
    )

    while response.choices[0].finish_reason == "tool_calls":
        response_message = response.choices[0].message

        # print("Assistant Response:")
        # print(response_message)
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures, names, args = [], [], []

            for tool in response_message.tool_calls:
                print("Invoking Recommended Function call:")
                print(tool.function)
                print()

                function_name = tool.function.name

                names.append(function_name)
                function_to_call = available_functions[function_name]
                function_args = json.loads(tool.function.arguments)
                args.append(function_args)
                futures.append(executor.submit(function_to_call,**function_args))

        for future, name, arg in zip(futures, names, args):
            function_response = future.result()


            print(f"Output of function call of {MANGETA}{name}{RESET} with {dict_to_expression_string(arg)}:")
            print(function_response)
            print()

            messages.append(
                {
                    "role": response_message.role,
                    "function_call": {
                        "name": tool.function.name,
                        "arguments": tool.function.arguments,
                    },
                    "content": None,
                }
            )
            messages.append(
                {
                    "role": "function",
                    "name": function_name,
                    "content": function_response,
                }
            )

        print("Messages in next request:")
        for message in messages:
            print(message)
        print()

        response = client.chat.completions.create(
            messages=messages,
            tools=tools,
            tool_choice="auto",
            model=userdata.get('AZURE_MODEL_NAME'),
            temperature=1e-4,
        )
        # ends with


    messages.append(
        {
            "role": response.choices[0].message.role,
            "content": response.choices[0].message.content,
        }
    )
    return response

In [ ]:
tool_question_answer = {
    "type": "function",
    "function": {
        "name": "answer_user_question",
        "description": "Answer aa concise question from user",
        "parameters": {
            "type": "object",
            "properties": {
                "concise_user_question": {
                    "type": "string",
                    "description":
"""A question from user, which is a concise question.
If the user ask a complex question, this should be split into to different ones.""",
                },
            },
            "required": ["concise_user_question"],
        },
    },
}


In [ ]:
tools = [
    tool_question_answer,
]

def answer_user_question(concise_user_question):
    response = query_engine.query(concise_user_question)
    return str(response)

available_functions = {
    "answer_user_question": answer_user_question,
}

In [ ]:
from IPython.display import Markdown
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        # "content": "List the revenue of Alphabet and Tesla", # 5 seconds with k=5
        # "content": "Show the tables of Tesla", # 7 seconds
        # "content": "What was the net cash provided by operating activities in Q4-2022?", # 6 seconds
        # "content": "How did reductions in workforce and the change in server life affect Alphabet Inc.'s profitability in the first quarter of 2023, considering their impact on operating expenses and depreciation, and how do these factors relate to the company's strategy to maintain long-term growth and optimize its cost base?", # 9 seconds
        # "content": "Tell me the acountant info of the first quarter for Tesla and Alphabet, be detailed", # 7 seconds
        "content": "What are the specific software updates included in the latest version of Tesla's Full Self-Driving (FSD) system?",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
print(assistant_response.choices[0].message)
print("Conversation complete!")
Markdown(next_messages[-1]['content'].replace('$','\$'))

Invoking Recommended Function call:
Function(arguments='{"concise_user_question":"What are the specific software updates included in the latest version of Tesla\'s Full Self-Driving (FSD) system?"}', name='answer_user_question')

Output of function call of answer_user_question with concise_user_question=What are the specific software updates included in the latest version of Tesla's Full Self-Driving (FSD) system?:
The specific software updates included in the latest version of Tesla's Full Self-Driving (FSD) system are not detailed in the provided information.

Messages in next request:
{'role': 'system', 'content': '\nAssistant is a helpful assistant that helps users get answers to questions about a Neo4jGraph.\nAssistant has access to several tools and sometimes you may need to call multiple tools in sequence to get answers for your users.\n\n'}
{'role': 'user', 'content': "What are the specific software updates included in the latest version of Tesla's Full Self-Driving (FSD) syste

I don't have the specific details of the latest software updates for Tesla's Full Self-Driving (FSD) system. For the most accurate and up-to-date information, I recommend checking Tesla's official release notes or their website.

In [ ]:
# len(response.source_nodes)

3

In [ ]:
triplet_source_ids = list(response.metadata.keys())
triplet_source_ids

['2224caa0-6f33-4a8c-914f-d1c9a87d8991',
 '8b98192c-2ea2-462f-b930-cdcb83885f53',
 'b3de9a4d-1bd4-41ca-83b5-d21e017e6068']

In [ ]:
# [r.node.dict()['text'] for r in response.source_nodes]

In [ ]:
# response.

In [ ]:
# print(graph.run(f"""MATCH (c)
# RETURN id(c) as ID_OF_CHUNK_NODE, c.triplet_source_id as triplet_source_id
# """).to_data_frame().to_markdown()) # .text

In [ ]:
# print(graph.run(f"""MATCH (c:Chunk)
# WITH apoc.convert.fromJsonMap(c._node_content) AS content, c.text AS text,c
# WHERE content.id_ IN {triplet_source_ids}
# RETURN id(c) as ID_OF_CHUNK_NODE
# """).to_data_frame().to_markdown()) # .text

|    |   ID_OF_CHUNK_NODE |
|---:|-------------------:|
|  0 |                 31 |
|  1 |                 36 |
|  2 |                 40 |
|  3 |                 42 |
|  4 |                 44 |
|  5 |                 47 |
|  6 |                 63 |


In [ ]:
# print(graph.run(f"""MATCH (c)-[r:GENERATES]->(t)
# RETURN DISTINCT id(c) as ID_OF_CHUNK_NODE, r.triplet_source_id as triplet_source_id
# """).to_data_frame().to_markdown()) # .text

|    |   ID_OF_CHUNK_NODE | triplet_source_id                    |
|---:|-------------------:|:-------------------------------------|
|  0 |                 68 | 9ba4e7f9-d0a5-4cd0-a555-ab49d871e1c8 |
|  1 |                 68 | d8f06812-25d0-4119-b8fe-f2f4cc990466 |
|  2 |                 91 | cbaf889d-039e-458a-b296-d2e0935ade3f |
|  3 |                 68 | 4bd8cbcf-3b51-403c-904a-83c062a6783a |
|  4 |                 68 | ee4149c4-e39f-4d72-999d-08b4defb3d71 |
|  5 |                 68 | 25bf822b-828e-4862-b9c2-f804db542ca1 |
|  6 |                 68 | 6fe967fe-2f0c-410a-af02-ed195f86dfc9 |
|  7 |                200 | fa04c11d-7d15-4073-86f1-7c3c4118ef6f |
|  8 |                251 | 99c4190b-9447-45f4-90ae-48968c2f0c84 |
|  9 |                255 | 49d3e93a-1174-4d84-8862-225dbf9d72f7 |
| 10 |                258 | 49d3e93a-1174-4d84-8862-225dbf9d72f7 |
| 11 |                223 | 3f7845ab-7788-41cf-bf3a-6631e80bcfa9 |
| 12 |                223 | af3dac42-cfbf-49f4-8d98-829c476997

In [ ]:
# import pandas as pd
# results = []

# for triplet_source_id in triplet_source_ids:
#     query = f"""
#     MATCH (c)-[r]->(t)
#     WHERE c.triplet_source_id = '{triplet_source_id}'
#       AND t.triplet_source_id = '{triplet_source_id}'
#       AND r.triplet_source_id = '{triplet_source_id}'
#     RETURN c.name, type(r), t.name
#     """
#     result = graph.run(query).to_data_frame()
#     results.append(result)

# # Concatenate all DataFrames into a single DataFrame
# combined_results = pd.concat(results, ignore_index=True).drop_duplicates()

# # Convert to markdown and print
# print(combined_results.to_markdown())

In [ ]:
response = query_engine.query("Tell me the revenue of Alphabet")
print(str(response))

For the quarter ended March 31, 2023, Alphabet's total revenues were $69.787 billion.


In [ ]:
# triplet_source_ids = list(response.metadata.keys())
# triplet_source_ids

['2766132c-d5b6-4621-853c-9ff6a3a8cd8b',
 '99c4190b-9447-45f4-90ae-48968c2f0c84',
 'c5914a5b-9ee3-4dc9-b944-703352b996d0',
 'ecc217b1-aba4-4087-a53c-00f69717ee62',
 'bdcc383e-dd69-4c10-938c-691dcd0125d5',
 'fa04c11d-7d15-4073-86f1-7c3c4118ef6f']

In [ ]:
# import pandas as pd
# results = []

# for triplet_source_id in triplet_source_ids:
#     query = f"""
#     MATCH (c)-[r]->(t)
#     WHERE c.triplet_source_id = '{triplet_source_id}'
#       AND t.triplet_source_id = '{triplet_source_id}'
#       AND r.triplet_source_id = '{triplet_source_id}'
#     RETURN c.name, type(r), t.name
#     """
#     result = graph.run(query).to_data_frame()
#     results.append(result)

# # Concatenate all DataFrames into a single DataFrame
# combined_results = pd.concat(results, ignore_index=True).drop_duplicates()

# # Convert to markdown and print
# print(combined_results.to_markdown())

|    | c.name                                                   | type(r)       | t.name                                                   |
|---:|:---------------------------------------------------------|:--------------|:---------------------------------------------------------|
|  0 | gaap revenues                                            | REPORTED      | non gaap percentage change in constant currency revenues |
|  1 | gaap percentage change in revenues                       | REPORTED      | non gaap percentage change in constant currency revenues |
|  2 | non gaap percentage change in constant currency revenues | GENERATES     | non gaap percentage change in constant currency revenues |
|  5 | alphabet inc.                                            | REPORTED      | consolidated statements of income                        |
|  6 | alphabet inc.                                            | INCURS        | research and development                                 |
|  7 | alphab

In [ ]:
response = query_engine.query("Tell me the revenue of Tesla and Alphabet")
print(str(response))

Tesla's total revenue for Q1 2023 was $23.329 billion. Alphabet's revenue for Q1 2023 was not explicitly stated in the provided information.


In [ ]:
triplet_source_ids = list(response.metadata.keys())
triplet_source_ids

['7175fb84-d7ab-4831-bec9-aeee30227d99',
 '2c51eb99-c24e-40dd-8072-8e6f58104ae5',
 'ecc217b1-aba4-4087-a53c-00f69717ee62',
 '55551a8b-828e-4f00-ae71-9fa00f986e12',
 '7da89fc9-def1-4521-8ed1-23e2238030fb',
 '2766132c-d5b6-4621-853c-9ff6a3a8cd8b',
 '1ad188e5-9a8b-4fc9-a7d1-842ad50bd322',
 'f2a0cb63-a261-46a0-a2ed-f0f449471d20']

In [ ]:
import pandas as pd
results = []

for triplet_source_id in triplet_source_ids:
    query = f"""
    MATCH (c)-[r]->(t)
    WHERE c.triplet_source_id = '{triplet_source_id}'
      AND t.triplet_source_id = '{triplet_source_id}'
      AND r.triplet_source_id = '{triplet_source_id}'
    RETURN c.name, type(r), t.name
    """
    result = graph.run(query).to_data_frame()
    results.append(result)

# Concatenate all DataFrames into a single DataFrame
combined_results = pd.concat(results, ignore_index=True).drop_duplicates()

# Convert to markdown and print
print(combined_results.to_markdown())

|    | c.name                                                   | type(r)       | t.name                                                   |
|---:|:---------------------------------------------------------|:--------------|:---------------------------------------------------------|
|  0 | adjusted ebitda                                          | REPORTED      | income statement                                         |
|  1 | depreciation amortization impairment                     | REPORTED      | income statement                                         |
|  2 | provision for income taxes                               | REPORTED      | income statement                                         |
|  3 | tesla                                                    | HAS_ASSET     | battery cells                                            |
|  4 | tesla                                                    | HAS_ASSET     | factories                                                |
|  5 | tesla 

In [ ]:
response = query_engine.query("show me the table of revenues, trafficyt Acquisition Costs of Alphabet")
print(str(response))

Here is the table of revenues, Traffic Acquisition Costs (TAC), and number of employees for Alphabet:

**Quarter Ended March 31,**
| (in millions, except for number of employees; unaudited) | 2022 | 2023 |
|----------------------------------------------------------|------|------|
| Google Search & other                                    | $39,618 | $40,359 |
| YouTube ads                                              | $6,869 | $6,693 |
| Google Network                                           | $8,174 | $7,496 |
| Google advertising                                       | $54,661 | $54,548 |
| Google other                                             | $6,811 | $7,413 |
| Google Services total                                    | $61,472 | $61,961 |
| Google Cloud                                             | $5,821 | $7,454 |
| Other Bets                                               | $440 | $288 |
| Hedging gains (losses)                                   | $278 | $84 |
| Total rev

In [ ]:
response = query_engine.query("what is Free Cash Flow of 2022 and 2023 in the first Quarter of Tesla")
print(str(response))

In the first quarter of 2022, Tesla's free cash flow was 2,228 million USD. In the first quarter of 2023, it was 441 million USD.


In [ ]:
response = query_engine.query("show me the table of Free cash flow of Tesla, be detailed and extensed and tell me the sources")
print(str(response))

Here is the detailed and extensive table of Free Cash Flow for Tesla:

| Period   | Free Cash Flow (non-GAAP) in millions of USD |
|----------|---------------------------------------------|
| 3Q-2019  | 371                                         |
| 4Q-2019  | 1,013                                       |
| 1Q-2020  | (895)                                       |
| 2Q-2020  | 418                                         |
| 3Q-2020  | 1,395                                       |
| 4Q-2020  | 1,868                                       |
| 1Q-2021  | 293                                         |
| 2Q-2021  | 619                                         |
| 3Q-2021  | 1,328                                       |
| 4Q-2021  | 2,775                                       |
| 1Q-2022  | 2,228                                       |
| 2Q-2022  | 621                                         |
| 3Q-2022  | 3,297                                       |
| 4Q-2022  | 1,420                         

In [ ]:
response = query_engine.query("Tell me the acountant info of the first quarter for Tesla and Alphabet, be detailed")
print(str(response))

For the first quarter of 2023, Tesla reported the following financial details:

- **Total automotive revenues**: $19,963 million
- **Energy generation and storage revenue**: $1,529 million
- **Services and other revenue**: $1,837 million
- **Total revenues**: $23,329 million
- **Total gross profit**: $4,511 million
- **Total GAAP gross margin**: 19.3%
- **Operating expenses**: $1,847 million
- **Income from operations**: $2,664 million
- **Operating margin**: 11.4%
- **Adjusted EBITDA**: $4,267 million
- **Adjusted EBITDA margin**: 18.3%
- **Net income attributable to common stockholders (GAAP)**: $2,513 million
- **Net income attributable to common stockholders (non-GAAP)**: $2,931 million
- **EPS attributable to common stockholders, diluted (GAAP)**: $0.73
- **EPS attributable to common stockholders, diluted (non-GAAP)**: $0.85
- **Net cash provided by operating activities**: $2,513 million
- **Capital expenditures**: $2,072 million
- **Free cash flow**: $441 million
- **Cash, cash e

In [ ]:
triplet_source_ids = list(response.metadata.keys())

In [ ]:
graph = Graph(url, auth=(username, password))

In [ ]:
print(graph.run(f"""MATCH (c:Chunk)
WITH apoc.convert.fromJsonMap(c._node_content) AS content, c.text AS text,c
WHERE content.id_ IN {triplet_source_ids}
RETURN id(c)
""").to_data_frame().to_markdown()) # .text

|    |   id(c) |
|---:|--------:|
|  0 |      31 |
|  1 |      36 |
|  2 |      40 |
|  3 |      42 |
|  4 |      44 |
|  5 |      47 |
|  6 |      63 |


In [ ]:
# from IPython.display import Markdown, display
# query = "Tell me the powerpack of Tesla"
# response = query_engine.query(query)
# Markdown(str(response))

In [ ]:
# triplet_source_ids = list(set([node.dict()['node']['relationships']['1']['node_id'] for node in response.source_nodes]))
# triplet_source_ids

In [ ]:
# nodes_relations = index.as_retriever(include_text=False, similarity_top_k=20).retrieve(query)
# triplet_source_ids = list(set([node.dict()['node']['relationships']['1']['node_id'] for node in nodes_relations]))
# triplet_source_ids

In [ ]:
import pandas as pd
results = []

for triplet_source_id in triplet_source_ids:
    query = f"""
    MATCH (c)-[r]->(t)
    WHERE c.triplet_source_id = '{triplet_source_id}'
      AND t.triplet_source_id = '{triplet_source_id}'
      AND r.triplet_source_id = '{triplet_source_id}'
    RETURN c.name, type(r), t.name
    """
    result = graph.run(query).to_data_frame()
    results.append(result)

# Concatenate all DataFrames into a single DataFrame
combined_results = pd.concat(results, ignore_index=True)

# Convert to markdown and print
print(combined_results.to_markdown())

|    | c.name                                         | type(r)       | t.name                                                                           |
|---:|:-----------------------------------------------|:--------------|:---------------------------------------------------------------------------------|
|  0 | tesla                                          | LED_BY        | elon musk                                                                        |
|  1 | alphabet inc.                                  | REPORTED      | consolidated statements of income                                                |
|  2 | alphabet inc.                                  | INCURS        | research and development                                                         |
|  3 | alphabet inc.                                  | INCURS        | cost of revenues                                                                 |
|  4 | other income expense                           | REPORTED      

In [ ]:
query = f"""
MATCH (c)-[r]->(t)
WHERE c.triplet_source_id IN {triplet_source_ids}
  AND t.triplet_source_id IN {triplet_source_ids}
  AND r.triplet_source_id IN {triplet_source_ids}
RETURN c.name, type(r),t.name
"""
graph.run(query).to_data_frame()

,c.name,type(r),t.name
0,adjusted ebitda,REPORTED,income statement
1,adjusted ebitda,REPORTED,trailing twelve months
2,general and administrative,INCURS,"1,076"
3,total operating expenses,INCURS,"1,076"
4,interest income,GENERATES,213
5,interest expense,INCURS,"1,076"
6,interest expense,REPORTED,"1,076"
7,interest expense,REPORTED,income statement
8,debt and finance leases,HAS_LIABILITY,current portion
9,total liabilities,HAS_LIABILITY,redeemable noncontrolling interests


In [ ]:
#@title display the graph

query = f"""
MATCH (c)-[r]->(t)
WHERE c.triplet_source_id IN {triplet_source_ids}
  AND t.triplet_source_id IN {triplet_source_ids}
  AND r.triplet_source_id IN {triplet_source_ids}
RETURN c AS node,r,t
"""
g.show_cypher(query)

NameError: name 'g' is not defined

# FC

In [ ]:
# Can add system prompting to guide the model to call functions and perform in specific ways
system_prompt = f"""
Assistant is a helpful assistant that helps users get answers to questions about a Neo4jGraph.
Assistant has access to several tools and sometimes you may need to call multiple tools in sequence to get answers for your users.

"""
next_messages = [
    {
        "role": "system",
        "content": system_prompt,
    }]

In [ ]:
from openai import AzureOpenAI
from google.colab import userdata
import json
from pprint import pprint

client = AzureOpenAI(
    api_key=userdata.get('AZURE_API_KEY'),
    api_version=userdata.get('AZURE_API_VERSION'),
    azure_endpoint=userdata.get('AZURE_BASE_URL')
)

import concurrent.futures

BLUE,GREEN,MANGETA,RESET = '\033[94m', '\033[92m', '\033[95m', '\033[0m'
def dict_to_expression_string(dictionary):
    return ' '.join(f'{BLUE}{key}{RESET}={GREEN}{value}{RESET}' for key, value in dictionary.items())

def run_multiturn_conversation(messages, tools, available_functions):
    response = client.chat.completions.create(
        messages=messages,
        tools=tools,
        tool_choice="auto",
        model=userdata.get('AZURE_MODEL_NAME'),
        temperature=1e-4,
    )

    while response.choices[0].finish_reason == "tool_calls":
        response_message = response.choices[0].message

        # print("Assistant Response:")
        # print(response_message)
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures, names, args = [], [], []

            for tool in response_message.tool_calls:
                print("Invoking Recommended Function call:")
                print(tool.function)
                print()

                function_name = tool.function.name

                names.append(function_name)
                function_to_call = available_functions[function_name]
                function_args = json.loads(tool.function.arguments)
                args.append(function_args)
                futures.append(executor.submit(function_to_call,**function_args))

        for future, name, arg in zip(futures, names, args):
            function_response = future.result()


            print(f"Output of function call of {MANGETA}{name}{RESET} with {dict_to_expression_string(arg)}:")
            print(function_response)
            print()

            messages.append(
                {
                    "role": response_message.role,
                    "function_call": {
                        "name": tool.function.name,
                        "arguments": tool.function.arguments,
                    },
                    "content": None,
                }
            )
            messages.append(
                {
                    "role": "function",
                    "name": function_name,
                    "content": function_response,
                }
            )

        print("Messages in next request:")
        for message in messages:
            print(message)
        print()

        response = client.chat.completions.create(
            messages=messages,
            tools=tools,
            tool_choice="auto",
            model=userdata.get('AZURE_MODEL_NAME'),
            temperature=1e-4,
        )
        # ends with


    messages.append(
        {
            "role": response.choices[0].message.role,
            "content": response.choices[0].message.content,
        }
    )
    return response

In [ ]:
tool_question_answer = {
    "type": "function",
    "function": {
        "name": "answer_user_question",
        "description": "Answer aa concise question from user",
        "parameters": {
            "type": "object",
            "properties": {
                "concise_user_question": {
                    "type": "string",
                    "description":
"""A question from user, which is a concise question.
If the user ask a complex question, this should be split into to different ones.""",
                },
            },
            "required": ["concise_user_question"],
        },
    },
}


In [ ]:
tools = [
    tool_question_answer,
]

def answer_user_question(concise_user_question):
    response = query_engine.query(concise_user_question)
    return str(response)

available_functions = {
    "answer_user_question": answer_user_question,
}